In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pip install numpy

In [ ]:
RAW = "data/raw"
OUT = "data/processed"

In [ ]:
# ---------------------------------------------------------------
# 1. LOAD + BASIC CLEANING
# ---------------------------------------------------------------
patients = pd.read_csv(f"{RAW}/patients.csv")
services_weekly = pd.read_csv(f"{RAW}/services_weekly.csv")
staff_schedule = pd.read_csv(f"{RAW}/staff_schedule.csv")

In [ ]:
# dedup + standardize text
for df in [patients, services_weekly, staff_schedule]:
    df.drop_duplicates(inplace=True)

In [ ]:
patients["service"] = patients["service"].str.strip().str.lower()
services_weekly["service"] = services_weekly["service"].str.strip().str.lower()
staff_schedule["service"] = staff_schedule["service"].str.strip().str.lower()
staff_schedule["role"] = staff_schedule["role"].str.strip().str.lower()

In [ ]:
# dates
patients["arrival_date"] = pd.to_datetime(patients["arrival_date"], errors="coerce")
patients["departure_date"] = pd.to_datetime(patients["departure_date"], errors="coerce")
assert patients["arrival_date"].isnull().sum() == 0
assert patients["departure_date"].isnull().sum() == 0

In [ ]:
# outlier checks
assert (patients["age"] >= 0).all() and (patients["age"] <= 120).all()
patients["length_of_stay_days"] = (patients["departure_date"] - patients["arrival_date"]).dt.days
assert (patients["length_of_stay_days"] >= 0).all()

In [ ]:
patients["arrival_week"] = patients["arrival_date"].dt.isocalendar().week.astype(int)
patients["year"] = patients["arrival_date"].dt.year
patients["month"] = patients["arrival_date"].dt.month
patients["quarter"] = patients["arrival_date"].dt.quarter
patients["day_of_week"] = patients["arrival_date"].dt.day_name()

In [ ]:
# no duplicate patient_id -> every stay is a distinct admission
assert patients["patient_id"].is_unique
patients["admission_id"] = "ADM-" + patients["patient_id"].str.replace("PAT-", "", regex=False)
patients.rename(columns={"satisfaction": "patient_satisfaction_score"}, inplace=True)

In [ ]:
# ---------------------------------------------------------------
# 2. HOSPITAL OVERVIEW  (grain: 1 row = 1 admission)
# ---------------------------------------------------------------
hospital_overview = patients[[
    "admission_id", "patient_id", "name", "age", "service",
    "arrival_date", "departure_date", "length_of_stay_days",
    "patient_satisfaction_score", "year", "month", "quarter", "day_of_week"
]].rename(columns={
    "service": "department_name",
    "arrival_date": "admission_date",
    "departure_date": "discharge_date",
})
# no repeat patient_ids exist in source -> no true readmissions observable
hospital_overview["readmission_flag"] = 0
hospital_overview.to_csv(f"{OUT}/hospital_overview_dataset.csv", index=False)

In [ ]:
# ---------------------------------------------------------------
# 3. PATIENT FLOW  (grain: 1 row = 1 movement event)
#    NOTE: source has no inter-department transfer data - only a
#    single service + arrival/departure date per patient. Modeled
#    as a 2-event flow (Admission, Discharge) per admission.
#    hour_of_day / shift / is_peak_hour are NOT derivable (no
#    timestamps in source) - left out rather than fabricated.
# ---------------------------------------------------------------
adm_events = hospital_overview[["admission_id", "patient_id", "department_name", "admission_date", "year", "month", "day_of_week"]].copy()
adm_events["movement_type"] = "Admission"
adm_events["movement_date"] = adm_events["admission_date"]
adm_events["duration_in_department_hours"] = hospital_overview["length_of_stay_days"] * 24

In [ ]:
dis_events = hospital_overview[["admission_id", "patient_id", "department_name", "discharge_date", "year", "month", "day_of_week"]].copy()
dis_events["movement_type"] = "Discharge"
dis_events["movement_date"] = dis_events["discharge_date"]
dis_events["duration_in_department_hours"] = np.nan

In [ ]:
patient_flow = pd.concat([
    adm_events.drop(columns=["admission_date"]),
    dis_events.drop(columns=["discharge_date"]),
], ignore_index=True)
patient_flow.sort_values(["admission_id", "movement_date"], inplace=True)
patient_flow.insert(0, "movement_id", ["MOV-" + str(i+1).zfill(5) for i in range(len(patient_flow))])
patient_flow.to_csv(f"{OUT}/patient_flow_dataset.csv", index=False)

In [ ]:
# ---------------------------------------------------------------
# 4. DEPARTMENT ANALYTICS  (grain: week + department)
#    Built from patients.csv directly (ground truth), NOT from
#    services_weekly/hospital_insights_summary - those two datasets
#    do not reconcile with patients.csv totals or with each other
#    (see methodology.md). available_beds from services_weekly is
#    kept as a labeled *reference* field only.
# ---------------------------------------------------------------
dept = hospital_overview.groupby(["year", "arrival_week" if "arrival_week" in hospital_overview else "month", "department_name"]) if False else None

In [ ]:
hospital_overview["arrival_week"] = patients["arrival_week"]
dept_analytics = hospital_overview.groupby(["year", "arrival_week", "department_name"]).agg(
    patients_admitted_count=("admission_id", "count"),
    avg_length_of_stay_days=("length_of_stay_days", "mean"),
    avg_satisfaction_score=("patient_satisfaction_score", "mean"),
).reset_index().rename(columns={"arrival_week": "week"})

In [ ]:
dept_analytics["patients_discharged_count"] = dept_analytics["patients_admitted_count"]  # same stay window closes within data horizon for most cases
dept_analytics["readmission_count"] = 0
dept_analytics["readmission_rate_pct"] = 0.0

In [ ]:
# bring in available_beds as a labeled reference (separate source, weekly)
beds_ref = services_weekly[["week", "service", "available_beds", "event"]].rename(
    columns={"service": "department_name", "available_beds": "available_beds_ref", "event": "event_ref"}
)
dept_analytics = dept_analytics.merge(beds_ref, on=["week", "department_name"], how="left")
dept_analytics["bed_occupancy_rate_ref_pct"] = (
    dept_analytics["patients_admitted_count"] / dept_analytics["available_beds_ref"] * 100
).round(1)

In [ ]:
dept_analytics.to_csv(f"{OUT}/department_analytics_dataset.csv", index=False)

In [ ]:
# ---------------------------------------------------------------
# 5. RESOURCE UTILIZATION  (grain: week + department + resource_type)
#    staff_schedule.csv used as authoritative staff roster/presence
#    (its per-service headcounts of 34/39/28/25 match
#    hospital_insights_summary.staff_count; staff.csv's 32/29/27/22
#    do not, so staff.csv is excluded - see methodology.md).
#    NO equipment data exists in any source file - left undocumented
#    columns out entirely rather than fabricated.
# ---------------------------------------------------------------
roster_capacity = staff_schedule.groupby(["service", "role"])["staff_id"].nunique().reset_index(name="total_units_available")

In [ ]:
staff_weekly = staff_schedule[staff_schedule["present"] == 1].groupby(
    ["week", "service", "role"]
).size().reset_index(name="units_in_use")

In [ ]:
staff_res = staff_weekly.merge(roster_capacity, on=["service", "role"], how="left")
staff_res["utilization_rate_pct"] = (staff_res["units_in_use"] / staff_res["total_units_available"] * 100).round(1)
staff_res.rename(columns={"service": "department_name", "role": "resource_type"}, inplace=True)
staff_res["resource_category"] = "staff"

In [ ]:
# bed resource rows from services_weekly (capacity=available_beds, in_use=weekly admissions as proxy)
bed_res = services_weekly[["week", "service", "available_beds", "patients_admitted"]].rename(
    columns={"service": "department_name", "available_beds": "total_units_available", "patients_admitted": "units_in_use"}
)
bed_res["resource_type"] = "bed"
bed_res["resource_category"] = "bed"
bed_res["utilization_rate_pct"] = (bed_res["units_in_use"] / bed_res["total_units_available"] * 100).round(1)

In [ ]:
resource_utilization = pd.concat([
    staff_res[["week", "department_name", "resource_type", "resource_category", "total_units_available", "units_in_use", "utilization_rate_pct"]],
    bed_res[["week", "department_name", "resource_type", "resource_category", "total_units_available", "units_in_use", "utilization_rate_pct"]],
], ignore_index=True)
resource_utilization.insert(0, "resource_utilization_id", ["RES-" + str(i+1).zfill(5) for i in range(len(resource_utilization))])
resource_utilization.to_csv(f"{OUT}/resource_utilization_dataset.csv", index=False)

In [ ]:
print("All 4 processed tables written to", OUT)
print("hospital_overview:", hospital_overview.shape)
print("patient_flow:", patient_flow.shape)
print("department_analytics:", dept_analytics.shape)
print("resource_utilization:", resource_utilization.shape)

All 4 processed tables written to data/processed
hospital_overview: (1000, 15)
patient_flow: (2000, 10)
department_analytics: (208, 12)
resource_utilization: (628, 8)
